<a href="https://colab.research.google.com/github/ChanZH0525/Salary-Prediction-Prototype-Model/blob/main/Job_Salary_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# 1. LOAD ONLY NECESSARY COLUMNS
cols_to_load = [
    'title',
    'description',
    'location',
    'work_type',
    'normalized_salary',
    'pay_period',
    'formatted_experience_level',
    'remote_allowed',  # Could be useful
    'skills_desc'      # Only 245 non-null, but might help
]

df = pd.read_csv('postings.csv',
                 usecols=cols_to_load,
                 on_bad_lines='warn',
                 engine='python')

print(f"Loaded {len(df)} rows with {len(df.columns)} columns")

# 2. FILTER FOR YEARLY SALARIES ONLY
df = df[df['pay_period'] == 'YEARLY'].copy()
df.drop('pay_period', axis=1, inplace=True)

print(f"After filtering yearly: {len(df)} rows")

# 3. HANDLE MISSING VALUES
print("\nMissing values before cleaning:")
print(df.isnull().sum())

# Fill missing experience levels
df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Not Specified')

# Fill missing skills description
df['skills_desc'] = df['skills_desc'].fillna('')

# Create remote flag from remote_allowed
df['is_remote'] = df['remote_allowed'].notna().astype(int)
df.drop('remote_allowed', axis=1, inplace=True)

# 4. TEXT CLEANING FUNCTION
def clean_text(text):
    """Basic text cleaning for job descriptions"""
    if pd.isna(text):
        return ""
    # Remove excessive whitespace
    text = ' '.join(text.split())
    # Remove special characters (keep basic punctuation)
    text = text.replace('\\n', ' ').replace('\\r', ' ')
    return text

# Apply cleaning
df['description'] = df['description'].apply(clean_text)
df['title'] = df['title'].apply(clean_text)
df['skills_desc'] = df['skills_desc'].apply(clean_text)

# 5. EXTRACT LOCATION FEATURES
def extract_location_features(location):
    """Extract city and state from location"""
    parts = location.split(',')
    city = parts[0].strip() if len(parts) > 0 else location
    state = parts[1].strip() if len(parts) > 1 else 'Unknown'
    return pd.Series({'city': city, 'state': state})

location_features = df['location'].apply(extract_location_features)
df = pd.concat([df, location_features], axis=1)

# 6. REMOVE OUTLIERS IN SALARY
# Keep only reasonable salary ranges (adjust based on your domain knowledge)
salary_percentiles = df['normalized_salary'].quantile([0.01, 0.99])
print(f"\nSalary range: ${salary_percentiles[0.01]:,.0f} - ${salary_percentiles[0.99]:,.0f}")

df = df[
    (df['normalized_salary'] >= salary_percentiles[0.01]) &
    (df['normalized_salary'] <= salary_percentiles[0.99])
]

print(f"After removing salary outliers: {len(df)} rows")

# 7. CREATE ENHANCED TEXT INPUT
df['text_input'] = (
    "[TITLE] " + df['title'] +
    " [LOCATION] " + df['location'] +
    " [CITY] " + df['city'] +
    " [STATE] " + df['state'] +
    " [WORK_TYPE] " + df['work_type'] +
    " [EXPERIENCE] " + df['formatted_experience_level'] +
    " [REMOTE] " + df['is_remote'].map({0: 'No', 1: 'Yes'})
)

# Add skills if available
df['text_input'] += df['skills_desc'].apply(
    lambda x: f" [SKILLS] {x[:200]}" if x else ""
)

# Add description (truncated)
df['text_input'] += " [DESCRIPTION] " + df['description'].str.slice(0, 500)

# 8. TARGET TRANSFORMATION
df['log_salary'] = np.log1p(df['normalized_salary'])

# 9. FINAL CLEANUP
# Remove any rows with empty text_input
df = df[df['text_input'].str.len() > 50]  # Minimum meaningful length

# 10. DISPLAY RESULTS
print(f"\nFinal dataset: {len(df)} rows")
print("\nSample of text_input:")
print(df[['title', 'normalized_salary', 'text_input']].head(2))
print("\nData types:")
print(df.dtypes)
print("\nSalary statistics:")
print(df['normalized_salary'].describe())

Loaded 123849 rows with 9 columns
After filtering yearly: 20628 rows

Missing values before cleaning:
title                             0
description                       1
location                          0
remote_allowed                17212
formatted_experience_level     5058
skills_desc                   20487
work_type                         0
normalized_salary                 0
dtype: int64

Salary range: $44 - $337,500
After removing salary outliers: 20217 rows

Final dataset: 20217 rows

Sample of text_input:
                                               title  normalized_salary  \
2                        Assitant Restaurant Manager            55000.0   
3  Senior Elder Law / Trusts and Estates Associat...           157500.0   

                                          text_input  
2  [TITLE] Assitant Restaurant Manager [LOCATION]...  
3  [TITLE] Senior Elder Law / Trusts and Estates ...  

Data types:
title                          object
description                    o

In [2]:
import re
import numpy as np

def extract_years_experience(df):
    """Extract required years of experience from job descriptions"""

    def find_years(text):
        if pd.isna(text):
            return np.nan

        text = str(text).lower()

        # Multiple patterns to catch different phrasings
        patterns = [
            # "5+ years of experience", "3 years experience"
            r'(\d+)\+?\s*years?\s+(?:of\s+)?(?:related\s+)?(?:relevant\s+)?experience',
            # "minimum of 5 years", "minimum 5 years"
            r'minimum\s+(?:of\s+)?(\d+)\+?\s*years?',
            # "at least 5 years"
            r'at\s+least\s+(\d+)\+?\s*years?',
            # "5-7 years experience" - take the lower bound
            r'(\d+)\s*-\s*\d+\s*years?\s+(?:of\s+)?experience',
            # "5+ years in", "5 years of"
            r'(\d+)\+?\s*years?\s+(?:in|of|working)',
            # "experience: 5 years" or "experience of 5 years"
            r'experience\s*(?:of|:)?\s*(\d+)\+?\s*years?',
            # Just "5+ years" when near experience context
            r'(\d+)\s*\+\s*years?',
        ]

        years_found = []
        for pattern in patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                years = float(match)
                if 0 <= years <= 50:  # Reasonable range
                    years_found.append(years)

        if years_found:
            # Use the most common requirement (usually minimum)
            return min(years_found)  # Minimum years required
        return np.nan

    print("Extracting years of experience from descriptions...")
    df['years_experience'] = df['description'].apply(find_years)

    # Statistics before imputation
    extracted = df['years_experience'].notna().sum()
    print(f"✓ Found explicit experience for {extracted} jobs ({extracted/len(df)*100:.1f}%)")

    if extracted > 0:
        print(f"  Mean: {df['years_experience'].mean():.1f} years")
        print(f"  Median: {df['years_experience'].median():.1f} years")
        print(f"  Range: {df['years_experience'].min():.0f}-{df['years_experience'].max():.0f} years")

    # Now impute missing values based on job title seniority
    print("\nImputing missing experience from job titles...")

    # Title-based experience mapping
    title_to_exp = {
        # Entry level
        'internship': 0.5, 'intern': 0.5,
        'junior': 1.5, 'jr.': 1.5, 'jr ': 1.5,
        'entry': 1.5, 'entry-level': 1.5, 'entry level': 1.5,
        'trainee': 0.5, 'apprentice': 1,

        # Mid level
        'associate': 3, 'mid-level': 4, 'mid level': 4,
        'intermediate': 3,

        # Senior level
        'senior': 7, 'sr.': 7, 'sr ': 7,
        'senior-level': 7, 'senior level': 7,
        'lead': 8, 'principal': 10, 'staff': 10,

        # Management
        'manager': 6, 'supervisor': 5, 'team lead': 6,
        'head': 10, 'head of': 10,
        'director': 12, 'senior director': 14,
        'vp': 15, 'vice president': 15, 'svp': 18,
        'chief': 18, 'c-suite': 18, 'executive': 18,
        'ceo': 20, 'cto': 18, 'cfo': 18, 'coo': 18,
        'president': 18, 'partner': 15, 'owner': 10,

        # Specialized roles
        'architect': 10, 'scientist': 8, 'engineer': 5,
        'analyst': 3, 'consultant': 5, 'specialist': 4,
        'coordinator': 3, 'representative': 2, 'clerk': 1,
        'assistant': 2, 'technician': 3, 'operator': 2,
        'therapist': 3, 'nurse': 3, 'teacher': 3,
        'chef': 4, 'cook': 2, 'driver': 2, 'sales': 3,
        'agent': 2, 'officer': 3, 'accountant': 3,
        'attorney': 5, 'lawyer': 5, 'paralegal': 2,
    }

    # Apply title heuristics
    imputed_count = 0
    for title_word, exp_guess in sorted(title_to_exp.items(), key=lambda x: len(x[0]), reverse=True):
        # Sort by length to match longer titles first (e.g., "senior director" before "director")
        mask = df['years_experience'].isna() & df['title'].str.lower().str.contains(title_word, na=False)
        if mask.sum() > 0:
            df.loc[mask, 'years_experience'] = exp_guess
            imputed_count += mask.sum()

    print(f"✓ Imputed experience for {imputed_count} jobs from titles")

    # Fill remaining with overall median
    median_exp = df['years_experience'].median()
    remaining = df['years_experience'].isna().sum()
    df['years_experience'] = df['years_experience'].fillna(median_exp)
    print(f"✓ Filled remaining {remaining} jobs with median value ({median_exp:.1f} years)")

    # Final statistics
    print(f"\n=== FINAL EXPERIENCE DISTRIBUTION ===")
    print(df['years_experience'].describe())

    # Check correlation with salary
    corr = df['normalized_salary'].corr(df['years_experience'])
    print(f"\n💰 Correlation between years_experience and salary: {corr:.3f}")

    # Show distribution
    print(f"\n=== SALARY BY EXPERIENCE BUCKET ===")
    df['exp_bucket'] = pd.cut(df['years_experience'],
                               bins=[0, 2, 5, 8, 12, 20, 50],
                               labels=['0-2 yrs', '3-5 yrs', '6-8 yrs', '9-12 yrs', '13-20 yrs', '20+ yrs'])
    print(df.groupby('exp_bucket', observed=False)['normalized_salary'].agg(['mean', 'median', 'count']))

    return df

# Run extraction
df = extract_years_experience(df)

# Show some examples
print("\n=== EXAMPLES ===")
sample = df[df['years_experience'].notna()].head(5)
for _, row in sample.iterrows():
    print(f"\nTitle: {row['title']}")
    print(f"Years Experience: {row['years_experience']:.1f}")
    print(f"Salary: ${row['normalized_salary']:,.0f}")

Extracting years of experience from descriptions...
✓ Found explicit experience for 12283 jobs (60.8%)
  Mean: 4.9 years
  Median: 4.0 years
  Range: 0-50 years

Imputing missing experience from job titles...
✓ Imputed experience for 6735 jobs from titles
✓ Filled remaining 1199 jobs with median value (4.0 years)

=== FINAL EXPERIENCE DISTRIBUTION ===
count    20217.000000
mean         4.993125
std          3.960502
min          0.000000
25%          3.000000
50%          4.000000
75%          6.000000
max         50.000000
Name: years_experience, dtype: float64

💰 Correlation between years_experience and salary: 0.224

=== SALARY BY EXPERIENCE BUCKET ===
                     mean    median  count
exp_bucket                                
0-2 yrs      91068.688776   79000.0   3967
3-5 yrs     109567.835283  100000.0  10796
6-8 yrs     124088.883124  118950.0   3204
9-12 yrs    161075.910539  160000.0   1401
13-20 yrs   133934.184121  121125.0    637
20+ yrs      90277.400769   80000.0

In [3]:
# ============================================
# FIX 1: Cap unrealistic experience values
# ============================================

# Cap at 20 years (anything above is usually a parsing error)
df['years_experience'] = df['years_experience'].clip(upper=20)

# Re-check correlation
corr = df['normalized_salary'].corr(df['years_experience'])
print(f"Corrected correlation: {corr:.3f}")

print("\n=== CORRECTED SALARY BY EXPERIENCE ===")
df['exp_bucket'] = pd.cut(df['years_experience'],
                           bins=[0, 2, 5, 8, 12, 20],
                           labels=['0-2 yrs', '3-5 yrs', '6-8 yrs', '9-12 yrs', '13-20 yrs'])
print(df.groupby('exp_bucket', observed=False)['normalized_salary'].agg(['mean', 'median', 'count']))

# ============================================
# FIX 2: Better text input with experience
# ============================================

def create_enhanced_text(row):
    """Create text that makes experience VERY explicit for BERT"""

    parts = []

    # Start with experience - most important for salary!
    parts.append(f"YEARS OF EXPERIENCE: {row['years_experience']:.0f}")

    # Title
    parts.append(f"TITLE: {row['title']}")

    # Experience level category
    if row['formatted_experience_level'] != 'Not Specified':
        parts.append(f"LEVEL: {row['formatted_experience_level']}")

    # Work type
    parts.append(f"TYPE: {row['work_type']}")

    # Location
    parts.append(f"LOCATION: {row['location']}")

    # Remote
    if row['is_remote'] == 1:
        parts.append("REMOTE: Yes")

    # Skills
    if row['skills_desc'] and len(str(row['skills_desc'])) > 20:
        parts.append(f"SKILLS: {str(row['skills_desc'])[:300]}")

    # Description
    if row['description']:
        parts.append(f"DESCRIPTION: {str(row['description'])[:400]}")

    return " | ".join(parts)

df['text_input'] = df.apply(create_enhanced_text, axis=1)

print("\n=== SAMPLE ENHANCED INPUT ===")
print(df['text_input'].iloc[0])

Corrected correlation: 0.260

=== CORRECTED SALARY BY EXPERIENCE ===
                     mean    median  count
exp_bucket                                
0-2 yrs      91068.688776   79000.0   3967
3-5 yrs     109567.835283  100000.0  10796
6-8 yrs     124088.883124  118950.0   3204
9-12 yrs    161075.910539  160000.0   1401
13-20 yrs   125930.440506  110000.0    780

=== SAMPLE ENHANCED INPUT ===
YEARS OF EXPERIENCE: 20 | TITLE: Assitant Restaurant Manager | TYPE: FULL_TIME | LOCATION: Cincinnati, OH | SKILLS: We are currently accepting resumes for FOH - Asisstant Restaurant Management with a strong focus on delivering high quality customer service. Prefer 1 to 3 years FOH management experience. Candidate should be a self-starter, proactive, attentive to details and like developing others. Must have a str | DESCRIPTION: The National Exemplar is accepting applications for an Assistant Restaurant Manager. We offer highly competitive wages, healthcare, paid time off, complimentary dining

In [4]:
# 7. STRATIFIED TRAIN/VAL/TEST SPLIT
from sklearn.model_selection import train_test_split

# First split: train+val vs test
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=pd.qcut(df['normalized_salary'], q=5, labels=False)  # Stratify by salary quintiles
)

# Second split: train vs val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.176,  # 0.15 / 0.85 ≈ 0.176 to get ~15% of total
    random_state=42,
    stratify=pd.qcut(train_val_df['normalized_salary'], q=5, labels=False)
)

print(f"\nData splits:")
print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

# 8. SAMPLE THE ENHANCED TEXT
print("\n=== SAMPLE ENHANCED TEXT INPUT ===")
print(df['text_input'].iloc[0])
print("\n" + "="*80)


Data splits:
Train: 14159 (70.0%)
Validation: 3025 (15.0%)
Test: 3033 (15.0%)

=== SAMPLE ENHANCED TEXT INPUT ===
YEARS OF EXPERIENCE: 20 | TITLE: Assitant Restaurant Manager | TYPE: FULL_TIME | LOCATION: Cincinnati, OH | SKILLS: We are currently accepting resumes for FOH - Asisstant Restaurant Management with a strong focus on delivering high quality customer service. Prefer 1 to 3 years FOH management experience. Candidate should be a self-starter, proactive, attentive to details and like developing others. Must have a str | DESCRIPTION: The National Exemplar is accepting applications for an Assistant Restaurant Manager. We offer highly competitive wages, healthcare, paid time off, complimentary dining privileges and bonus opportunities. We are a serious, professional, long-standing neighborhood restaurant with over 41 years of service. If you are looking for a long-term fit with a best in class organization then you should apply 



In [5]:
# 1. INSTALL & IMPORT (if needed)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import numpy as np

# 2. CHOOSE MODEL - OPTIONS TO TRY
# Option A: DistilBERT (faster, lighter)
#model_name = "distilbert-base-uncased"

# Option B: For better performance (slower but more accurate)
# model_name = "bert-base-uncased"

# Option C: For understanding job titles better
model_name = "jjzha/jobbert-base-cased"  # Pre-trained on job descriptions!

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1  # Regression task
)

# 3. TOKENIZATION FUNCTION
def tokenize_function(examples):
    return tokenizer(
        examples["text_input"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

# 4. CONVERT TO HUGGING FACE DATASETS
train_dataset = Dataset.from_pandas(train_df[['text_input', 'log_salary']])
val_dataset = Dataset.from_pandas(val_df[['text_input', 'log_salary']])
test_dataset = Dataset.from_pandas(test_df[['text_input', 'log_salary']])

# Rename target column
train_dataset = train_dataset.rename_column("log_salary", "labels")
val_dataset = val_dataset.rename_column("log_salary", "labels")
test_dataset = test_dataset.rename_column("log_salary", "labels")

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# 5. ENHANCED METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.flatten()

    # Log-space metrics
    r2_log = r2_score(labels, predictions)
    mse_log = mean_squared_error(labels, predictions)

    # Convert back to USD for interpretable metrics
    real_labels = np.expm1(labels)
    real_predictions = np.expm1(predictions)

    mae_usd = mean_absolute_error(real_labels, real_predictions)
    rmse_usd = np.sqrt(mean_squared_error(real_labels, real_predictions))

    # Calculate accuracy within different thresholds
    within_10k = np.mean(np.abs(real_labels - real_predictions) < 10000)
    within_20k = np.mean(np.abs(real_labels - real_predictions) < 20000)

    return {
        "r2": r2_log,
        "mae_usd": mae_usd,
        "rmse_usd": rmse_usd,
        "accuracy_10k": within_10k,
        "accuracy_20k": within_20k,
    }



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/603 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: jjzha/jobbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider train

Map:   0%|          | 0/14159 [00:00<?, ? examples/s]

Map:   0%|          | 0/3025 [00:00<?, ? examples/s]

Map:   0%|          | 0/3033 [00:00<?, ? examples/s]

In [7]:
# ============================================
# ADD EARLY STOPPING + BETTER TRAINING CONFIG
# ============================================

from transformers import EarlyStoppingCallback, DataCollatorWithPadding

# Fixed training arguments with early stopping
training_args = TrainingArguments(
    output_dir="./salary_model_results",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,  # Will stop early if no improvement
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae_usd",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    report_to="none",  # Disable wandb/tensorboard if not needed
)

# Create trainer WITH early stopping callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,  # Stop if no improvement for 3 evaluations
            early_stopping_threshold=0.001,  # Minimum change to count as improvement
        )
    ],
)

# Check GPU
print(f"Using device: {trainer.args.device}")
print(f"FP16 training: {trainer.args.fp16}")
print(f"Early stopping patience: 3 epochs")
print(f"Metric for best model: mae_usd (lower is better)")

# Start training
trainer.train()

# Show which epoch was best
print(f"\nBest model checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best metric value: {trainer.state.best_metric:.4f}")

# Save the model
trainer.save_model("./salary_model_final")

# 8. EVALUATE ON TEST SET
print("\n=== FINAL TEST SET EVALUATION ===")
test_results = trainer.evaluate(test_dataset)
print("\nTest Results:")
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Using device: cuda:0
FP16 training: True
Early stopping patience: 3 epochs
Metric for best model: mae_usd (lower is better)


Epoch,Training Loss,Validation Loss,R2,Mae Usd,Rmse Usd,Accuracy 10k,Accuracy 20k
1,0.667605,0.548011,0.170272,27908.916016,38272.429763,0.248926,0.480331
2,0.654650,0.516821,0.217495,24462.646484,35379.538493,0.326281,0.562645
3,0.438904,0.537367,0.186387,26936.574219,35871.157885,0.251240,0.479669
4,0.392361,0.501537,0.240637,22721.121094,32834.643168,0.340496,0.588430
5,0.317253,0.539490,0.183173,29512.304688,38221.866830,0.205620,0.413223
6,0.180647,0.495367,0.249977,27124.244141,36067.966064,0.240992,0.470744
7,0.152398,0.495362,0.249986,25380.945312,34327.255177,0.269421,0.518678


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Best model checkpoint: ./salary_model_results/checkpoint-3540
Best metric value: 22721.1211


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== FINAL TEST SET EVALUATION ===



Test Results:
eval_loss: 0.5623
eval_r2: 0.2217
eval_mae_usd: 23167.6758
eval_rmse_usd: 33581.5187
eval_accuracy_10k: 0.3106
eval_accuracy_20k: 0.5885
eval_runtime: 26.4667
eval_samples_per_second: 114.5970
eval_steps_per_second: 7.1790
epoch: 7.0000


In [ ]:
# 9. ANALYZE PREDICTION ERRORS
def analyze_predictions():
    """Get detailed predictions for analysis"""
    predictions = trainer.predict(test_dataset)

    results_df = test_df.copy()
    results_df['predicted_salary'] = np.expm1(predictions.predictions.flatten())
    results_df['prediction_error'] = results_df['predicted_salary'] - results_df['normalized_salary']
    results_df['abs_error'] = np.abs(results_df['prediction_error'])

    print("\n=== WORST PREDICTIONS ===")
    worst = results_df.nlargest(5, 'abs_error')[['title', 'normalized_salary', 'predicted_salary', 'prediction_error']]
    print(worst)

    print("\n=== BEST PREDICTIONS ===")
    best = results_df.nsmallest(5, 'abs_error')[['title', 'normalized_salary', 'predicted_salary', 'prediction_error']]
    print(best)

    print("\n=== ERROR BY EXPERIENCE LEVEL ===")
    print(results_df.groupby('formatted_experience_level')['abs_error'].mean().sort_values())

    return results_df

results_df = analyze_predictions()


=== WORST PREDICTIONS ===
                                                   title  normalized_salary  \
17229                               Sales Representative           295000.0   
18704                        Strategic Account Executive           308000.0   
34093  Family Medicine Physician - $205k to $378k Sal...            15000.0   
67594     Principal Wealth Manager - Investment Advisory              300.0   
84644                Project Finance Associate - 1890298           337500.0   

       predicted_salary  prediction_error  
17229      63734.691406    -231265.308594  
18704     101848.343750    -206151.656250  
34093     215614.062500     200614.062500  
67594     197858.500000     197558.500000  
84644     142511.859375    -194988.140625  

=== BEST PREDICTIONS ===
                               title  normalized_salary  predicted_salary  \
116737     People Programs Associate            70000.0      69998.960938   
40778                   Art Director           120000.